# YARA RAG Pipeline — Génération de Règles YARA via RAG
**Sujet 1 : Génération de règles YARA **

## Architecture
- **Retrieval** : FAISS (dense) + BM25 (sparse) + Rerank + Agentique
- **Embeddings** : `sentence-transformers/all-MiniLM-L6-v2`
- **LLM** : `mistralai/Mistral-7B-Instruct-v0.1`
- **Interface** : Gradio
- **Évaluation** : métriques par type de RAG + benchmarking



## Cellule 1 — Installation des dépendances

In [ ]:
# ================================================================
# CELLULE 1 — Installation des dépendances
# ================================================================
# À exécuter une seule fois au démarrage du runtime Colab

!pip install -q faiss-cpu rank_bm25 sentence-transformers
!pip install -q transformers accelerate bitsandbytes
!pip install -q scikit-learn
!pip install -q gradio
!pip install -q pymupdf   # Pour l'ingestion PDF (optionnel)

print("Toutes les dépendances sont installées.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 82.0 MB/s eta 0:00:00
Toutes les dépendances sont installées.


## ⚙️ Cellule 2 — Imports & Configuration globale

In [ ]:
# ================================================================
# CELLULE 2 — Imports & Configuration globale
# ================================================================

import os, json, pickle, time, re
from pathlib import Path
from datetime import datetime
from collections import Counter
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Retrieval
import faiss
from rank_bm25 import BM25Okapi

# Embeddings
from sentence_transformers import SentenceTransformer, CrossEncoder

# LLM local
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# Evaluation
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

# Interface
import gradio as gr

# ----------------------------------------------------------------
# Chemins (compatibles Google Colab /content/)
# ----------------------------------------------------------------
BASE_DIR     = "/content/yara_rag"
DATASET_PATH = "/content/dataset_yara_propre_v2.json"
CACHE_DIR    = f"{BASE_DIR}/cache"
INDEX_PATH   = f"{CACHE_DIR}/faiss_index.bin"
EMBED_PATH   = f"{CACHE_DIR}/embeddings.npy"
DOCS_PATH    = f"{CACHE_DIR}/documents.pkl"
BM25_PATH    = f"{CACHE_DIR}/bm25_index.pkl"
MODEL_CACHE  = f"{CACHE_DIR}/models"

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(MODEL_CACHE, exist_ok=True)

# ----------------------------------------------------------------
# Modèles
# ----------------------------------------------------------------
EMBED_MODEL_NAME  = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# ================================================================
# 3 LLMs disponibles — décommenter celui à utiliser
# ================================================================

# Option 1 — TinyLlama (1.1B) — le plus léger, testing rapide
# LLM_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
# LLM_DIR  = f'{MODEL_CACHE}/tinyllama'

# Option 2 — Phi-2 (2.7B) — bon pour code/YARA
# LLM_NAME = 'microsoft/phi-2'
# LLM_DIR  = f'{MODEL_CACHE}/phi2'

# Option 3 — Mistral-7B — meilleure qualité (ACTIF)
LLM_NAME = 'mistralai/Mistral-7B-Instruct-v0.1'
LLM_DIR  = f'{MODEL_CACHE}/mistral'

# RAG params
TOP_K     = 3
CLUSTER_N = 6

# Singletons globaux
_embed_model  = None
_rerank_model = None
_llm_pipeline = None
_index        = None
_documents    = None
_bm25         = None

print(f"Configuration chargée")
print(f" GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU : {torch.cuda.get_device_name(0)}")
print(f"   LLM actif     : {LLM_NAME}")
print(f"   Répertoire    : {BASE_DIR}")

Configuration chargée
 GPU disponible : True
   GPU : Tesla T4
   LLM actif     : mistralai/Mistral-7B-Instruct-v0.1
   Répertoire    : /content/yara_rag


## Cellule 4 — Chargement du dataset & construction des documents

In [ ]:
# ================================================================
# CELLULE 4 — Chargement du dataset & construction des documents
# ================================================================

def charger_dataset(path=DATASET_PATH):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"[Dataset] {len(data)} règles chargées")
    return data


def construire_documents(data):
    """
    Chaque document = texte concaténé pour l'embedding
    description + tags + famille + extraits de la règle
    """
    documents = []
    for r in data:
        tags_str     = " ".join(r.get('tags', []))
        strings_yara = re.findall(r'\$\w+\s*=\s*"([^"]+)"', r.get('regle_yara', ''))
        strings_str  = " ".join(strings_yara[:5])

        texte = (
            f"{r['description']} "
            f"famille {r['famille']} "
            f"danger {r['danger']} "
            f"{tags_str} "
            f"{strings_str}"
        ).lower().strip()

        documents.append({
            "id"         : r['id'],
            "texte"      : texte,
            "description": r['description'],
            "famille"    : r['famille'],
            "danger"     : r['danger'],
            "tags"       : r.get('tags', []),
            "source"     : r.get('source', ''),
            "regle_yara" : r['regle_yara'],
            "raw"        : r
        })
    return documents


print("Fonctions de chargement définies.")

Fonctions de chargement définies.


## 🔍 Cellule 5 — Indexation FAISS + BM25 (avec cache)

In [ ]:
# ================================================================
# CELLULE 5 — Indexation FAISS + BM25
# ================================================================

def charger_ou_construire_index(data):
    """
    Vérifie si l'index existe déjà en cache.
    Si oui  → charge depuis le disque (rapide)
    Si non  → construit et sauvegarde (une seule fois)
    """
    documents = construire_documents(data)

    if all(Path(p).exists() for p in [INDEX_PATH, EMBED_PATH, DOCS_PATH, BM25_PATH]):
        print("[Index] Cache trouvé — chargement depuis disque...")
        index      = faiss.read_index(INDEX_PATH)
        embeddings = np.load(EMBED_PATH)
        with open(DOCS_PATH, 'rb') as f:
            documents = pickle.load(f)
        with open(BM25_PATH, 'rb') as f:
            bm25 = pickle.load(f)
        print(f"[Index] Chargé depuis cache OK — {len(documents)} documents")
        return index, embeddings, documents, bm25

    print("[Index] Pas de cache — construction de l'index...")
    print(f"[Embed] Chargement {EMBED_MODEL_NAME}...")
    embed_model = SentenceTransformer(EMBED_MODEL_NAME, cache_folder=MODEL_CACHE)

    textes     = [d['texte'] for d in documents]
    print(f"[Embed] Génération des embeddings pour {len(textes)} documents...")
    embeddings = embed_model.encode(textes, show_progress_bar=True, convert_to_numpy=True)
    embeddings = embeddings.astype(np.float32)
    faiss.normalize_L2(embeddings)

    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    tokenized = [t.split() for t in textes]
    bm25      = BM25Okapi(tokenized)

    faiss.write_index(index, INDEX_PATH)
    np.save(EMBED_PATH, embeddings)
    with open(DOCS_PATH, 'wb') as f:
        pickle.dump(documents, f)
    with open(BM25_PATH, 'wb') as f:
        pickle.dump(bm25, f)

    print(f"[Index] Index construit et sauvegardé en cache — {len(documents)} documents")
    return index, embeddings, documents, bm25


print("Fonctions d'indexation définies.")

Fonctions d'indexation définies.


## Cellule 6 — Modèles d'embedding & reranking (singletons)

In [ ]:
# ================================================================
# CELLULE 6 — Modèles d'embedding & reranking
# ================================================================

def get_embed_model():
    global _embed_model
    if _embed_model is None:
        print(f"[Embed] Chargement {EMBED_MODEL_NAME}...")
        _embed_model = SentenceTransformer(EMBED_MODEL_NAME, cache_folder=MODEL_CACHE)
        print("[Embed] Modèle d'embedding chargé.")
    return _embed_model


def get_rerank_model():
    global _rerank_model
    if _rerank_model is None:
        print(f"[Rerank] Chargement {RERANK_MODEL_NAME}...")
        _rerank_model = CrossEncoder(RERANK_MODEL_NAME, max_length=512)
        print("[Rerank] Modèle de reranking chargé.")
    return _rerank_model


def encoder_requete(query):
    model = get_embed_model()
    vec   = model.encode([query], convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(vec)
    return vec


print("Fonctions d'embedding définies.")

Fonctions d'embedding définies.


## Cellule 7 — Retrieval (4 types : Classique, Hybride, Rerank, Agentique)

In [ ]:
# ================================================================
# CELLULE 7 — Retrieval (4 types)
# ================================================================

# ---- 7A. RAG Standard (Classique) ----------------------------
def retrieval_classique(query, index, documents, top_k=TOP_K):
    """Recherche sémantique pure via FAISS"""
    t0      = time.time()
    vec     = encoder_requete(query)
    scores, indices = index.search(vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(documents):
            doc = documents[idx].copy()
            doc['score']          = float(score)
            doc['retrieval_type'] = 'faiss'
            results.append(doc)
    return results, time.time() - t0


# ---- 7B. RAG Hybride (FAISS + BM25) ----------------------------
def retrieval_hybride(query, index, documents, bm25, top_k=TOP_K, alpha=0.5):
    """
    Combine FAISS (sémantique) + BM25 (lexical)
    alpha = poids FAISS (1-alpha = poids BM25)
    """
    t0 = time.time()

    vec = encoder_requete(query)
    faiss_scores, faiss_indices = index.search(vec, min(top_k * 3, len(documents)))

    tokens      = query.lower().split()
    bm25_scores = bm25.get_scores(tokens)

    def normalize(arr):
        mn, mx = arr.min(), arr.max()
        if mx - mn < 1e-9:
            return np.zeros_like(arr)
        return (arr - mn) / (mx - mn)

    faiss_norm = normalize(faiss_scores[0])
    bm25_norm  = normalize(bm25_scores)

    combined = {}
    for rank, (score, idx) in enumerate(zip(faiss_norm, faiss_indices[0])):
        if idx < len(documents):
            combined[idx] = alpha * score
    for idx, score in enumerate(bm25_norm):
        if idx in combined:
            combined[idx] += (1 - alpha) * score
        else:
            combined[idx]  = (1 - alpha) * score

    top_indices = sorted(combined, key=lambda x: combined[x], reverse=True)[:top_k]
    results = []
    for idx in top_indices:
        doc = documents[idx].copy()
        doc['score']          = combined[idx]
        doc['retrieval_type'] = 'hybride'
        results.append(doc)

    return results, time.time() - t0


# ---- 7C. RAG Rerank (FAISS + CrossEncoder) ---------------------
def retrieval_rerank(query, index, documents, top_k=TOP_K, candidates=15):
    """
    Étape 1 : FAISS récupère N candidats
    Étape 2 : CrossEncoder rerank les candidats
    """
    t0 = time.time()
    vec = encoder_requete(query)
    n_candidates = min(candidates, len(documents))
    scores, indices = index.search(vec, n_candidates)

    candidats = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(documents):
            candidats.append(documents[idx])

    reranker      = get_rerank_model()
    pairs         = [(query, c['texte']) for c in candidats]
    rerank_scores = reranker.predict(pairs)

    ranked  = sorted(zip(rerank_scores, candidats), key=lambda x: x[0], reverse=True)
    results = []
    for score, doc in ranked[:top_k]:
        d = doc.copy()
        d['score']          = float(score)
        d['retrieval_type'] = 'rerank'
        results.append(d)

    return results, time.time() - t0


# ---- 7D. RAG Agentique ----------------------------------------
def retrieval_agentic(query, index, documents, bm25, top_k=TOP_K):
    """
    Stratégie adaptative :
    - Si query courte (<5 mots)   → BM25 pur
    - Si query contient famille   → filtre par famille + FAISS
    - Sinon                       → Hybride standard
    """
    t0 = time.time()
    tokens = query.lower().split()

    familles_connues = ['ransomware','trojan','worm','spyware','dropper','rootkit','backdoor','exploit']
    famille_detectee = next((f for f in familles_connues if f in query.lower()), None)

    if famille_detectee:
        docs_filtres = [d for d in documents if d['famille'] == famille_detectee]
        if len(docs_filtres) >= top_k:
            embed_model  = get_embed_model()
            textes_filt  = [d['texte'] for d in docs_filtres]
            embeds_filt  = embed_model.encode(textes_filt, convert_to_numpy=True).astype(np.float32)
            faiss.normalize_L2(embeds_filt)
            idx_local    = faiss.IndexFlatIP(embeds_filt.shape[1])
            idx_local.add(embeds_filt)
            vec          = encoder_requete(query)
            scores, idxs = idx_local.search(vec, top_k)
            results = []
            for score, i in zip(scores[0], idxs[0]):
                if i < len(docs_filtres):
                    doc = docs_filtres[i].copy()
                    doc['score']          = float(score)
                    doc['retrieval_type'] = 'agentic_famille'
                    results.append(doc)
            return results, time.time() - t0

    if len(tokens) < 5:
        bm25_scores = bm25.get_scores(tokens)
        top_idxs    = np.argsort(bm25_scores)[::-1][:top_k]
        results = []
        for idx in top_idxs:
            doc = documents[idx].copy()
            doc['score']          = float(bm25_scores[idx])
            doc['retrieval_type'] = 'agentic_bm25'
            results.append(doc)
        return results, time.time() - t0

    results, latence = retrieval_hybride(query, index, documents, bm25, top_k)
    for r in results:
        r['retrieval_type'] = 'agentic_hybride'
    return results, latence


print("Fonctions de retrieval définies (FAISS, Hybride, Rerank, Agentique).")

Fonctions de retrieval définies (FAISS, Hybride, Rerank, Agentique).


## Cellule 8 — LLM local

In [ ]:
# ================================================================
# CELLULE 8 — LLM local
# ================================================================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

torch.cuda.empty_cache()

from transformers import BitsAndBytesConfig

def charger_llm_local():
    """Charge le LLM local une seule fois en mémoire"""
    global _llm_pipeline
    if _llm_pipeline is not None:
        return _llm_pipeline

    llm_path = LLM_DIR if Path(LLM_DIR).exists() else LLM_NAME
    print(f"[LLM] Chargement {LLM_NAME}...")
    print("      Cela peut prendre quelques minutes au premier lancement...")

    tokenizer = AutoTokenizer.from_pretrained(llm_path, cache_dir=MODEL_CACHE)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        llm_path,
        cache_dir=MODEL_CACHE,
        quantization_config=bnb_config,
        device_map="auto"
    )

    _llm_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    print("[LLM] Chargé en mémoire.")
    return _llm_pipeline


def generer_regle(prompt):
    """Appel LLM local"""
    t0 = time.time()
    try:
        llm   = charger_llm_local()
        out   = llm(prompt, return_full_text=False)
        texte = out[0]['generated_text'].strip()
        return texte, time.time() - t0
    except Exception as e:
        print(f"[LLM] Erreur : {e}")
        return None, time.time() - t0


print("Fonctions LLM définies.")
print(" Le modèle sera chargé automatiquement à la première génération.")

Fonctions LLM définies.
 Le modèle sera chargé automatiquement à la première génération.


## Cellule 9 — Construction du prompt & parsing de la réponse

In [ ]:
# ================================================================
# CELLULE 9 — Construction du prompt & parsing
# ================================================================

def construire_prompt(query, docs_recuperes):
    """
    Prompt enrichi avec les documents récupérés.
    Le LLM doit retourner : règle YARA + description + référence.
    """
    contexte = ""
    for i, doc in enumerate(docs_recuperes, 1):
        contexte += f"""
--- Document {i} (famille: {doc['famille']}, danger: {doc['danger']}) ---
Description : {doc['description']}
Tags        : {', '.join(doc.get('tags',[]))}
Règle YARA  :
{doc['regle_yara']}
"""

    prompt = f"""Tu es un expert en cybersécurité spécialisé dans la création de règles YARA.

CONTEXTE — Règles similaires trouvées dans la base de connaissances :
{contexte}

DEMANDE UTILISATEUR :
{query}

INSTRUCTIONS :
1. Génère une règle YARA valide et pertinente pour la demande
2. La règle doit avoir : meta (description, author, date), strings (au moins 3), condition logique
3. Après la règle, écris une DESCRIPTION claire en français expliquant ce que détecte la règle
4. Après la description, écris une REFERENCE indiquant la famille, le niveau de danger, et les techniques

FORMAT DE RÉPONSE OBLIGATOIRE :
```yara
rule NomDeLaRegle {{
    meta:
        description = "..."
        author      = "GRPP-2"
        date        = "2026"
    strings:
        $s1 = "..."
        ...
    condition:
        ...
}}
```

DESCRIPTION:
[Explication claire de ce que détecte la règle et comment elle fonctionne]

REFERENCE:
Famille: [famille] | Danger: [niveau] | Techniques: [liste]
"""
    return prompt


def parser_reponse(texte_llm):
    """Extrait règle, description, référence depuis la réponse du LLM"""
    if not texte_llm:
        return {"regle": "", "description": "", "reference": "", "raw": texte_llm}

    # Extraire la règle YARA
    regle = ""
    match = re.search(r'```(?:yara)?(.*?)```', texte_llm, re.DOTALL)
    if match:
        regle = match.group(1).strip()
    else:
        match2 = re.search(r'(rule\s+\w+\s*\{.*?\})', texte_llm, re.DOTALL)
        if match2:
            regle = match2.group(1).strip()

    # Extraire description
    description = ""
    match_desc  = re.search(r'DESCRIPTION:\s*(.*?)(?=REFERENCE:|$)', texte_llm, re.DOTALL | re.IGNORECASE)
    if match_desc:
        description = match_desc.group(1).strip()

    # Extraire référence
    reference  = ""
    match_ref  = re.search(r'REFERENCE:\s*(.*?)$', texte_llm, re.DOTALL | re.IGNORECASE)
    if match_ref:
        reference = match_ref.group(1).strip()

    return {
        "regle"       : regle,
        "description" : description,
        "reference"   : reference,
        "raw"         : texte_llm
    }


print("Fonctions de prompt et parsing définies.")

Fonctions de prompt et parsing définies.


## Cellule 10 — Pipeline RAG complet

In [ ]:
# ================================================================
# CELLULE 10 — Pipeline RAG complet
# ================================================================

def pipeline_rag(query, index, documents, bm25,
                 rag_type="hybride", top_k=TOP_K):
    """
    Pipeline complet :
    1. Encodage requête
    2. Retrieval (selon rag_type)
    3. Construction prompt enrichi
    4. Génération LLM
    5. Parsing réponse
    """
    print(f"\n[RAG] Query    : {query}")
    print(f"[RAG] Type     : {rag_type}")

    # Étape 2 — Retrieval
    dispatch = {
        "classique"   : lambda: retrieval_classique(query, index, documents, top_k),
        "hybride" : lambda: retrieval_hybride(query, index, documents, bm25, top_k),
        "rerank"  : lambda: retrieval_rerank(query, index, documents, top_k),
        "agentic" : lambda: retrieval_agentic(query, index, documents, bm25, top_k),
    }
    docs, t_retrieval = dispatch.get(rag_type, dispatch["hybride"])()

    print(f"[RAG] Récupéré : {len(docs)} documents (latence: {t_retrieval:.2f}s)")
    for d in docs:
        print(f"       → {d['id']} | {d['famille']} | score={d['score']:.3f}")

    # Étape 3 — Prompt
    prompt = construire_prompt(query, docs)

    # Étape 4 — Génération
    print("[LLM] Génération en cours...")
    texte_llm, t_generation = generer_regle(prompt)

    # Étape 5 — Parsing
    resultat = parser_reponse(texte_llm)
    resultat.update({
        'docs_recuperes': docs,
        'query'         : query,
        'rag_type'      : rag_type,
        't_retrieval'   : t_retrieval,
        't_generation'  : t_generation,
        'timestamp'     : datetime.now().isoformat(),
    })
    print(f"[LLM] Génération terminée (latence: {t_generation:.2f}s)")
    return resultat


def initialiser(dataset_path=DATASET_PATH):
    """Charge tout au démarrage — ne se fait qu'une seule fois"""
    global _index, _documents, _bm25
    if _index is not None:
        return _index, _documents, _bm25
    data       = charger_dataset(dataset_path)
    _index, _, _documents, _bm25 = charger_ou_construire_index(data)
    return _index, _documents, _bm25


def generer(query, rag_type="hybride"):
    """Fonction principale — appel simple depuis Gradio"""
    index, documents, bm25 = initialiser()
    return pipeline_rag(query, index, documents, bm25, rag_type=rag_type)


print("Pipeline RAG complet défini.")

Pipeline RAG complet défini.


In [ ]:
# ================================================================
# CELLULE 11 — Évaluation & Benchmarking
# ================================================================

QUERIES_TEST = [
    "Détecter un ransomware qui chiffre les fichiers avec AES",
    "Trouver un trojan qui ouvre un reverse shell",
    "Identifier un worm qui se propage via SMB",
    "Détecter un spyware qui enregistre les frappes clavier",
]


def evaluer_retrieval(query, docs_recuperes, documents):
    """Precision@k, MRR, diversité"""
    familles_query = [f for f in ['ransomware','trojan','worm','spyware','dropper','rootkit'] if f in query.lower()]

    pertinents = 0
    first_rank = 0
    for i, doc in enumerate(docs_recuperes):
        est_pertinent = (
            any(f in doc['famille'] for f in familles_query) or
            any(f in doc['texte']   for f in familles_query) or
            any(tag.lower() in query.lower() for tag in doc.get('tags', []))
        )
        if est_pertinent:
            pertinents += 1
            if first_rank == 0:
                first_rank = i + 1

    k           = len(docs_recuperes)
    precision_k = pertinents / k if k > 0 else 0
    mrr         = 1 / first_rank if first_rank > 0 else 0
    diversite   = len(set(d['famille'] for d in docs_recuperes)) / k if k > 0 else 0
    score_moyen = np.mean([d['score'] for d in docs_recuperes]) if docs_recuperes else 0

    return {
        "precision_k" : round(precision_k, 3),
        "mrr"         : round(mrr, 3),
        "diversite"   : round(diversite, 3),
        "score_moyen" : round(float(score_moyen), 3),
    }


def evaluer_generation(regle_generee):
    """Syntaxe, complétude, nb strings, complexité condition"""
    if not regle_generee:
        return {"syntaxe_valide": 0, "completude": 0, "nb_strings": 0, "complexite_cond": 0}

    syntaxe_valide = int(
        bool(re.search(r'rule\s+\w+', regle_generee)) and
        'strings:'   in regle_generee and
        'condition:' in regle_generee
    )
    completude  = round((int('meta:' in regle_generee) + int('strings:' in regle_generee) +
                         int('condition:' in regle_generee) + int('author' in regle_generee)) / 4, 2)
    nb_strings  = len(re.findall(r'\$\w+\s*=', regle_generee))
    cond_match  = re.search(r'condition:(.*?)(?=\}|$)', regle_generee, re.DOTALL)
    cond_text   = cond_match.group(1).strip() if cond_match else ""
    complexite  = len(cond_text.split())

    return {
        "syntaxe_valide"  : syntaxe_valide,
        "completude"      : completude,
        "nb_strings"      : nb_strings,
        "complexite_cond" : complexite,
    }


def benchmark_rag_types(index, documents, bm25, queries=None):
    """Compare les 4 types de RAG sur les queries de test"""
    if queries is None:
        queries = QUERIES_TEST

    rag_types = ["classique", "hybride", "rerank", "agentic"]
    rapport   = {rt: {"retrieval": [], "generation": [], "latences_ret": [], "latences_gen": []} for rt in rag_types}

    print("\n" + "="*60)
    print(" BENCHMARK RAG — Comparaison des 4 types")
    print("="*60)

    for query in queries:
        print(f"\nQuery : {query[:50]}...")
        for rt in rag_types:
            try:
                res   = pipeline_rag(query, index, documents, bm25, rag_type=rt)
                m_ret = evaluer_retrieval(query, res['docs_recuperes'], documents)
                m_gen = evaluer_generation(res['regle'])
                rapport[rt]['retrieval'].append(m_ret)
                rapport[rt]['generation'].append(m_gen)
                rapport[rt]['latences_ret'].append(res['t_retrieval'])
                rapport[rt]['latences_gen'].append(res['t_generation'])
                print(f"  [{rt:10s}] P@k={m_ret['precision_k']} MRR={m_ret['mrr']} syntaxe={m_gen['syntaxe_valide']}")
            except Exception as e:
                print(f"  [{rt:10s}] ERREUR : {e}")

    print("\n" + "="*60)
    print(" RÉSULTATS FINAUX")
    print("="*60)
    print(f"{'Type':12s} | {'P@k':6s} | {'MRR':6s} | {'Syntaxe':8s} | {'Complet':8s} | {'Lat_ret':8s}")
    print("-"*60)

    rapport_final = {}
    for rt in rag_types:
        if not rapport[rt]['retrieval']:
            continue
        avg_pk     = np.mean([m['precision_k']   for m in rapport[rt]['retrieval']])
        avg_mrr    = np.mean([m['mrr']            for m in rapport[rt]['retrieval']])
        avg_syn    = np.mean([m['syntaxe_valide'] for m in rapport[rt]['generation']])
        avg_comp   = np.mean([m['completude']     for m in rapport[rt]['generation']])
        avg_latret = np.mean(rapport[rt]['latences_ret'])
        rapport_final[rt] = {
            "precision_k"    : round(avg_pk, 3),
            "mrr"            : round(avg_mrr, 3),
            "syntaxe_valide" : round(avg_syn, 3),
            "completude"     : round(avg_comp, 3),
            "latence_ret_s"  : round(avg_latret, 3),
        }
        print(f"{rt:12s} | {avg_pk:6.3f} | {avg_mrr:6.3f} | {avg_syn:8.3f} | {avg_comp:8.3f} | {avg_latret:6.2f}s")

    with open(f"{CACHE_DIR}/benchmark_rapport.json", 'w') as f:
        json.dump(rapport_final, f, indent=2)
    print(f"\nRapport sauvegardé : {CACHE_DIR}/benchmark_rapport.json")
    return rapport_final


print("Fonctions d'évaluation et benchmarking définies.")

Fonctions d'évaluation et benchmarking définies.


## Cellule 12 — Clustering & Ingestion PDF

In [ ]:
# ================================================================
# CELLULE 12 — Clustering des règles & Ingestion PDF
# ================================================================

def clustering_regles(regles_generees, n_clusters=CLUSTER_N):
    """Groupe les règles générées par similarité sémantique"""
    if len(regles_generees) < n_clusters:
        n_clusters = max(2, len(regles_generees) // 2)

    textes = [r.get('description', '') + ' ' + r.get('regle', '') for r in regles_generees]
    vec    = TfidfVectorizer(max_features=100)
    X      = vec.fit_transform(textes).toarray()
    km     = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = km.fit_predict(X)

    clusters = {}
    for i, label in enumerate(labels):
        key = f"Cluster_{label}"
        if key not in clusters:
            clusters[key] = []
        clusters[key].append({
            "query"  : regles_generees[i].get('query', ''),
            "famille": regles_generees[i].get('rag_type', '')
        })
    return clusters, labels.tolist()


def ingerer_pdf(pdf_path, index, documents, bm25):
    """Ajoute un PDF comme nouvelle source de connaissance"""
    try:
        import fitz
    except ImportError:
        print("[PDF] PyMuPDF non installé. pip install pymupdf")
        return index, documents, bm25

    print(f"[PDF] Ingestion de {pdf_path}...")
    doc_pdf  = fitz.open(pdf_path)
    nouveaux = []

    for i, page in enumerate(doc_pdf):
        texte_page = page.get_text().strip()
        if len(texte_page) < 50:
            continue
        paragraphes = [p.strip() for p in texte_page.split('\n\n') if len(p.strip()) > 30]
        for j, para in enumerate(paragraphes[:5]):
            doc = {
                "id"         : f"PDF_{Path(pdf_path).stem}_p{i}_c{j}",
                "texte"      : para.lower(),
                "description": para[:200],
                "famille"    : "pdf_knowledge",
                "danger"     : "info",
                "tags"       : ["pdf", Path(pdf_path).stem],
                "source"     : "pdf",
                "regle_yara" : "",
                "raw"        : {"pdf": pdf_path, "page": i}
            }
            nouveaux.append(doc)

    if not nouveaux:
        print("[PDF] Aucun contenu extrait")
        return index, documents, bm25

    embed_model = get_embed_model()
    textes_new  = [d['texte'] for d in nouveaux]
    embeds_new  = embed_model.encode(textes_new, convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(embeds_new)
    index.add(embeds_new)
    documents.extend(nouveaux)

    tokenized_all = [d['texte'].split() for d in documents]
    bm25          = BM25Okapi(tokenized_all)

    faiss.write_index(index, INDEX_PATH)
    with open(DOCS_PATH, 'wb') as f:
        pickle.dump(documents, f)
    with open(BM25_PATH, 'wb') as f:
        pickle.dump(bm25, f)

    print(f"[PDF] {len(nouveaux)} chunks ajoutés. Total documents : {len(documents)}")
    return index, documents, bm25


print("Clustering et ingestion PDF définis.")

Clustering et ingestion PDF définis.


## Cellule 13 — Test rapide en ligne de commande (optionnel)
Permet de tester le pipeline sans l'interface Gradio.

In [ ]:
# ================================================================
# CELLULE 13 — Test rapide (optionnel, sans Gradio)
# ================================================================

print("=== YARA RAG Pipeline — Test rapide ===\n")

index, documents, bm25 = initialiser()

query  = "Détecter un ransomware qui chiffre les fichiers avec AES"
result = pipeline_rag(query, index, documents, bm25, rag_type="hybride")

print("\n" + "="*60)
print("RÈGLE YARA GÉNÉRÉE :")
print("="*60)
print(result['regle'] or "(règle vide — vérifier le LLM)")

print("\nDESCRIPTION :")
print(result['description'] or "(vide)")

print("\nRÉFÉRENCE :")
print(result['reference'] or "(vide)")

print(f"\n Retrieval : {result['t_retrieval']:.2f}s | Génération : {result['t_generation']:.2f}s")

=== YARA RAG Pipeline — Test rapide ===

[Dataset] 80 règles chargées
[Index] Pas de cache — construction de l'index...
[Embed] Chargement sentence-transformers/all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[LLM] Chargé en mémoire.


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[LLM] Génération terminée (latence: 457.50s)

RÈGLE YARA GÉNÉRÉE :
(règle vide — vérifier le LLM)

DESCRIPTION :
(vide)

RÉFÉRENCE :
(vide)

 Retrieval : 5.85s | Génération : 457.50s


---
## Cellule 14 — Benchmark LLM (modèle par modèle)
> Exécutez une fois par modèle. Commentez/décommentez dans la Cellule 2 pour changer de LLM.

In [ ]:
# ================================================================
# CELLULE 14 — Benchmark LLM (un modèle à la fois)
# ================================================================
# COMMENT UTILISER :
#   1. Dans Cellule 2, activez le LLM souhaité (TinyLlama, Phi-2, ou Mistral)
#   2. Exécutez cette cellule
#   3. Les résultats sont sauvegardés dans /content/yara_rag/cache/benchmark_llm_<nom>.json
#   4. Répétez pour chaque LLM
#   5. Cellule 16 compare tous les résultats sauvegardés
# ================================================================

import os, json, re, time
import numpy as np

QUERIES_BENCH_LLM = [
    "Détecter un ransomware qui chiffre les fichiers avec AES",
    "Trouver un trojan qui ouvre un reverse shell",
    "Identifier un spyware qui enregistre les frappes clavier",
    "Détecter un dropper utilisant PowerShell",
]

def evaluer_regle_qualite(regle, description, query):
    """
    Métriques de qualité de la règle générée :
    - syntaxe_valide  : a rule + strings + condition
    - completude      : a meta + strings + condition + author
    - nb_strings      : nombre de signatures
    - a_hex           : contient des patterns hexadécimaux
    - pertinence      : mots-clés de la query présents dans la règle
    - desc_longueur   : longueur de la description (qualité explication)
    """
    if not regle:
        return {"syntaxe_valide":0,"completude":0.0,"nb_strings":0,
                "a_hex":0,"pertinence":0.0,"desc_longueur":0,"score_global":0.0}

    syntaxe  = int(bool(re.search(r'rule\s+\w+', regle))
                   and 'strings:' in regle and 'condition:' in regle)
    complet  = round((int('meta:' in regle) + int('strings:' in regle) +
                      int('condition:' in regle) + int('author' in regle)) / 4, 2)
    nb_str   = len(re.findall(r'\$\w+\s*=', regle))
    a_hex    = int(bool(re.search(r'\{[\s0-9A-Fa-f?]+\}', regle)))

    # Pertinence : mots de la query retrouvés dans la règle
    mots_query = set(query.lower().split()) - {'un','une','les','des','qui','le','la','de','du'}
    mots_regle = set(regle.lower().split())
    pertinence = round(len(mots_query & mots_regle) / max(len(mots_query), 1), 2)

    desc_len = len(description) if description else 0

    # Score global /10
    score = (
        syntaxe     * 3.0 +
        complet     * 2.0 +
        min(nb_str / 5, 1.0) * 2.0 +
        a_hex       * 1.0 +
        pertinence  * 1.0 +
        min(desc_len / 300, 1.0) * 1.0
    )

    return {
        "syntaxe_valide" : syntaxe,
        "completude"     : complet,
        "nb_strings"     : nb_str,
        "a_hex"          : a_hex,
        "pertinence"     : pertinence,
        "desc_longueur"  : desc_len,
        "score_global"   : round(score, 2)
    }


def benchmark_llm_actif(index, documents, bm25, rag_type="hybride"):
    """
    Benchmark du LLM actuellement actif (défini dans Cellule 2).
    Sauvegarde les résultats pour comparaison ultérieure.
    """
    nom_llm    = LLM_NAME.split('/')[-1]
    resultats  = []

    print(f"\n{'='*60}")
    print(f" BENCHMARK LLM : {nom_llm}")
    print(f" RAG type      : {rag_type}")
    print(f" Queries       : {len(QUERIES_BENCH_LLM)}")
    print(f"{'='*60}")

    for i, query in enumerate(QUERIES_BENCH_LLM, 1):
        print(f"\n[{i}/{len(QUERIES_BENCH_LLM)}] {query[:55]}...")
        t0  = time.time()
        res = pipeline_rag(query, index, documents, bm25, rag_type=rag_type)
        t_total = time.time() - t0

        m_ret = evaluer_retrieval(query, res['docs_recuperes'], documents)
        m_gen = evaluer_regle_qualite(res['regle'], res['description'], query)

        entree = {
            "query"          : query,
            "llm"            : nom_llm,
            "rag_type"       : rag_type,
            "t_retrieval"    : round(res['t_retrieval'], 2),
            "t_generation"   : round(res['t_generation'], 2),
            "t_total"        : round(t_total, 2),
            "retrieval"      : m_ret,
            "generation"     : m_gen,
            "regle_apercu"   : res['regle'][:150] if res['regle'] else "",
        }
        resultats.append(entree)

        print(f"  Score global : {m_gen['score_global']}/10")
        print(f"  Syntaxe OK   : {bool(m_gen['syntaxe_valide'])}")
        print(f"  Nb strings   : {m_gen['nb_strings']}")
        print(f"  Latence LLM  : {res['t_generation']:.1f}s")

    # Moyennes
    moy_score  = round(np.mean([r['generation']['score_global'] for r in resultats]), 2)
    moy_syntax = round(np.mean([r['generation']['syntaxe_valide'] for r in resultats]), 2)
    moy_lat    = round(np.mean([r['t_generation'] for r in resultats]), 2)
    moy_pk     = round(np.mean([r['retrieval']['precision_k'] for r in resultats]), 2)

    resume = {
        "llm"              : nom_llm,
        "rag_type"         : rag_type,
        "nb_queries"       : len(resultats),
        "score_moyen"      : moy_score,
        "syntaxe_valide"   : moy_syntax,
        "latence_gen_moy"  : moy_lat,
        "precision_k_moy"  : moy_pk,
        "details"          : resultats
    }

    # Sauvegarder
    path_out = f"{CACHE_DIR}/benchmark_llm_{nom_llm}.json"
    with open(path_out, 'w', encoding='utf-8') as f:
        json.dump(resume, f, indent=2, ensure_ascii=False)

    print(f"\n{'='*60}")
    print(f" RÉSUMÉ — {nom_llm}")
    print(f"  Score moyen     : {moy_score}/10")
    print(f"  Syntaxe valide  : {moy_syntax*100:.0f}%")
    print(f"  Latence moy LLM : {moy_lat:.1f}s")
    print(f"  Précision@K     : {moy_pk}")
    print(f"  Sauvegardé      : {path_out}")
    print(f"{'='*60}")

    return resume


# Lancer le benchmark du LLM actif
index, documents, bm25 = initialiser()
bench_result = benchmark_llm_actif(index, documents, bm25, rag_type="hybride")


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 BENCHMARK LLM : Mistral-7B-Instruct-v0.1
 RAG type      : hybride
 Queries       : 4

[1/4] Détecter un ransomware qui chiffre les fichiers avec AE...

[RAG] Query    : Détecter un ransomware qui chiffre les fichiers avec AES
[RAG] Type     : hybride
[RAG] Récupéré : 3 documents (latence: 0.03s)
       → SYN_RAN_004 | ransomware | score=0.766
       → RAN_001 | ransomware | score=0.740
       → SYN_RAN_008 | ransomware | score=0.561
[LLM] Génération en cours...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[LLM] Génération terminée (latence: 90.22s)
  Score global : 0.0/10
  Syntaxe OK   : False
  Nb strings   : 0
  Latence LLM  : 90.2s

[2/4] Trouver un trojan qui ouvre un reverse shell...

[RAG] Query    : Trouver un trojan qui ouvre un reverse shell
[RAG] Type     : hybride
[RAG] Récupéré : 3 documents (latence: 0.01s)
       → TRO_005 | trojan | score=1.000
       → TRO_001 | trojan | score=0.439
       → TRO_003 | trojan | score=0.406
[LLM] Génération en cours...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[LLM] Génération terminée (latence: 43.94s)
  Score global : 9.4/10
  Syntaxe OK   : True
  Nb strings   : 6
  Latence LLM  : 43.9s

[3/4] Identifier un spyware qui enregistre les frappes clavie...

[RAG] Query    : Identifier un spyware qui enregistre les frappes clavier
[RAG] Type     : hybride
[RAG] Récupéré : 3 documents (latence: 0.01s)
       → SPY_001 | spyware | score=1.000
       → SYN_SPY_030 | spyware | score=0.446
       → SPY_005 | spyware | score=0.400
[LLM] Génération en cours...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[LLM] Génération terminée (latence: 32.86s)
  Score global : 9.12/10
  Syntaxe OK   : True
  Nb strings   : 5
  Latence LLM  : 32.9s

[4/4] Détecter un dropper utilisant PowerShell...

[RAG] Query    : Détecter un dropper utilisant PowerShell
[RAG] Type     : hybride
[RAG] Récupéré : 3 documents (latence: 0.01s)
       → DRO_001 | dropper | score=1.000
       → SYN_DRO_037 | dropper | score=0.406
       → SYN_DRO_039 | dropper | score=0.375
[LLM] Génération en cours...
[LLM] Génération terminée (latence: 62.95s)
  Score global : 8.25/10
  Syntaxe OK   : True
  Nb strings   : 6
  Latence LLM  : 62.9s

 RÉSUMÉ — Mistral-7B-Instruct-v0.1
  Score moyen     : 6.69/10
  Syntaxe valide  : 75%
  Latence moy LLM : 57.5s
  Précision@K     : 1.0
  Sauvegardé      : /content/yara_rag/cache/benchmark_llm_Mistral-7B-Instruct-v0.1.json


## Cellule 15 — Benchmark complet des 4 types de RAG
> Compare Classique, Hybride, Rerank, Agentique sur toutes les queries de test.

In [ ]:
# ================================================================
# CELLULE 15 — Benchmark RAG types (avec métriques complètes)
# ================================================================

def benchmark_rag_complet(index, documents, bm25, queries=None):
    """
    Benchmark des 4 types de RAG avec métriques :
    Retrieval  : Precision@K, MRR, diversité, score moyen
    Generation : syntaxe, complétude, nb_strings, score global
    Performance: latence retrieval, latence génération
    """
    if queries is None:
        queries = QUERIES_TEST

    rag_types = ["classique", "hybride", "rerank", "agentic"]
    rapport   = {rt: {"retrieval":[], "generation":[], "lat_ret":[], "lat_gen":[]} for rt in rag_types}

    print(f"\n{'='*65}")
    print(f" BENCHMARK 4 TYPES RAG — {len(queries)} queries × {len(rag_types)} types")
    print(f"{'='*65}")

    for query in queries:
        print(f"\nQuery : {query[:55]}...")
        for rt in rag_types:
            try:
                res   = pipeline_rag(query, index, documents, bm25, rag_type=rt)
                m_ret = evaluer_retrieval(query, res['docs_recuperes'], documents)
                m_gen = evaluer_regle_qualite(res['regle'], res['description'], query)
                rapport[rt]['retrieval'].append(m_ret)
                rapport[rt]['generation'].append(m_gen)
                rapport[rt]['lat_ret'].append(res['t_retrieval'])
                rapport[rt]['lat_gen'].append(res['t_generation'])
                print(f"  [{rt:8s}] P@k={m_ret['precision_k']} | score={m_gen['score_global']}/10 | lat_ret={res['t_retrieval']:.2f}s")
            except Exception as e:
                print(f"  [{rt:8s}] ERREUR : {e}")

    # Tableau résumé
    print(f"\n{'='*75}")
    print(f" RÉSULTATS FINAUX")
    print(f"{'='*75}")
    print(f"{'Type':10s} | {'P@K':5s} | {'MRR':5s} | {'Score/10':8s} | {'Syntaxe':8s} | {'Lat_ret':8s} | {'Lat_gen':8s}")
    print("-"*75)

    rapport_final = {}
    for rt in rag_types:
        if not rapport[rt]['retrieval']:
            continue
        avg_pk    = round(np.mean([m['precision_k']    for m in rapport[rt]['retrieval']]), 3)
        avg_mrr   = round(np.mean([m['mrr']            for m in rapport[rt]['retrieval']]), 3)
        avg_score = round(np.mean([m['score_global']   for m in rapport[rt]['generation']]), 2)
        avg_syn   = round(np.mean([m['syntaxe_valide'] for m in rapport[rt]['generation']]), 2)
        avg_lret  = round(np.mean(rapport[rt]['lat_ret']), 2)
        avg_lgen  = round(np.mean(rapport[rt]['lat_gen']), 2)

        rapport_final[rt] = {
            "precision_k"   : avg_pk,
            "mrr"           : avg_mrr,
            "score_global"  : avg_score,
            "syntaxe"       : avg_syn,
            "latence_ret_s" : avg_lret,
            "latence_gen_s" : avg_lgen,
        }
        print(f"{rt:10s} | {avg_pk:5.3f} | {avg_mrr:5.3f} | {avg_score:8.2f} | {avg_syn:8.2f} | {avg_lret:6.2f}s  | {avg_lgen:6.2f}s")

    print(f"{'='*75}")

    best_rt = max(rapport_final, key=lambda x: rapport_final[x]['score_global'])
    print(f"\nMeilleur type RAG (score global) : {best_rt} ({rapport_final[best_rt]['score_global']}/10)")

    # Sauvegarder
    path_out = f"{CACHE_DIR}/benchmark_rag_types.json"
    with open(path_out, 'w', encoding='utf-8') as f:
        json.dump(rapport_final, f, indent=2, ensure_ascii=False)
    print(f"   Sauvegardé : {path_out}")

    return rapport_final

index, documents, bm25 = initialiser()
rapport_rag = benchmark_rag_complet(index, documents, bm25)


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 BENCHMARK 4 TYPES RAG — 4 queries × 4 types

Query : Détecter un ransomware qui chiffre les fichiers avec AE...

[RAG] Query    : Détecter un ransomware qui chiffre les fichiers avec AES
[RAG] Type     : classique
[RAG] Récupéré : 3 documents (latence: 0.01s)
       → SYN_RAN_004 | ransomware | score=0.756
       → SYN_RAN_006 | ransomware | score=0.728
       → SYN_RAN_008 | ransomware | score=0.728
[LLM] Génération en cours...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[LLM] Génération terminée (latence: 52.66s)
  [classique] P@k=1.0 | score=0.0/10 | lat_ret=0.01s

[RAG] Query    : Détecter un ransomware qui chiffre les fichiers avec AES
[RAG] Type     : hybride
[RAG] Récupéré : 3 documents (latence: 0.01s)
       → SYN_RAN_004 | ransomware | score=0.766
       → RAN_001 | ransomware | score=0.740
       → SYN_RAN_008 | ransomware | score=0.561
[LLM] Génération en cours...
[LLM] Génération terminée (latence: 49.72s)
  [hybride ] P@k=1.0 | score=9.17/10 | lat_ret=0.01s

[RAG] Query    : Détecter un ransomware qui chiffre les fichiers avec AES
[RAG] Type     : rerank
[Rerank] Chargement cross-encoder/ms-marco-MiniLM-L-6-v2...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

...

[LLM] Génération terminée (latence: 35.22s)
  [classique] P@k=1.0 | score=9.2/10 | lat_ret=0.01s

[RAG] Query    : Détecter un spyware qui enregistre les frappes clavier
[RAG] Type     : hybride
[RAG] Récupéré : 3 documents (latence: 0.01s)
       → SPY_001 | spyware | score=1.000
       → SYN_SPY_030 | spyware | score=0.465
       → SYN_SPY_032 | spyware | score=0.405
[LLM] Génération en cours...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[LLM] Génération terminée (latence: 51.11s)
  [hybride ] P@k=1.0 | score=9.2/10 | lat_ret=0.01s

[RAG] Query    : Détecter un spyware qui enregistre les frappes clavier
[RAG] Type     : rerank
[RAG] Récupéré : 3 documents (latence: 0.02s)
       → SPY_001 | spyware | score=7.679
       → SYN_SPY_026 | spyware | score=3.137
       → SYN_SPY_025 | spyware | score=3.133
[LLM] Génération en cours...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[LLM] Génération terminée (latence: 38.46s)
  [rerank  ] P@k=1.0 | score=9.4/10 | lat_ret=0.02s

[RAG] Query    : Détecter un spyware qui enregistre les frappes clavier
[RAG] Type     : agentic
[RAG] Récupéré : 3 documents (latence: 0.02s)
       → SPY_001 | spyware | score=0.698
       → SYN_SPY_030 | spyware | score=0.630
       → SPY_005 | spyware | score=0.627
[LLM] Génération en cours...
[LLM] Génération terminée (latence: 47.86s)
  [agentic ] P@k=1.0 | score=5.7/10 | lat_ret=0.02s

 RÉSULTATS FINAUX
Type       | P@K   | MRR   | Score/10 | Syntaxe  | Lat_ret  | Lat_gen 
---------------------------------------------------------------------------
classique  | 1.000 | 1.000 |     6.48 |     0.75 |   0.01s  |  44.79s
hybride    | 1.000 | 1.000 |     6.68 |     0.75 |   0.01s  |  47.91s
rerank     | 0.917 | 1.000 |     9.12 |     1.00 |   3.17s  |  48.30s
agentic    | 1.000 | 1.000 |     7.90 |     0.75 |   0.02s  |  48.20s

Meilleur type RAG (score global) : rerank (9.12/10)
   Sauveg

## Cellule 16 — Comparaison des LLMs benchmarkés
> Exécutez après avoir benchmarké au moins 2 LLMs avec la Cellule 15.

In [ ]:
# ================================================================
# CELLULE 16 — Comparaison des LLMs benchmarkés
# ================================================================
import glob, json
import numpy as np

def comparer_llms():
    """Charge tous les benchmarks sauvegardés et affiche la comparaison"""
    fichiers = glob.glob(f"{CACHE_DIR}/benchmark_llm_*.json")

    if not fichiers:
        print("Aucun benchmark trouvé. Exécutez d'abord la Cellule 15.")
        return

    print(f"\n{'='*65}")
    print(f" COMPARAISON DES LLMs ({len(fichiers)} modèles benchmarkés)")
    print(f"{'='*65}")
    print(f"{'LLM':30s} | {'Score/10':8s} | {'Syntaxe':8s} | {'Lat(s)':7s} | {'P@K':5s}")
    print("-"*65)

    rapports = []
    for f in sorted(fichiers):
        with open(f) as fp:
            r = json.load(fp)
        rapports.append(r)
        print(f"{r['llm']:30s} | {r['score_moyen']:8.2f} | "
              f"{r['syntaxe_valide']*100:6.0f}%  | "
              f"{r['latence_gen_moy']:7.1f} | {r['precision_k_moy']:5.3f}")

    print(f"{'='*65}")

    if len(rapports) >= 2:
        meilleur = max(rapports, key=lambda x: x['score_moyen'])
        plus_rapide = min(rapports, key=lambda x: x['latence_gen_moy'])
        print(f"\nMeilleur score    : {meilleur['llm']} ({meilleur['score_moyen']}/10)")
        print(f"Plus rapide        : {plus_rapide['llm']} ({plus_rapide['latence_gen_moy']:.1f}s)")

    # Analyse détaillée query par query
    print(f"\n{'─'*65}")
    print(" DÉTAIL PAR QUERY")
    print(f"{'─'*65}")
    for r in rapports:
        print(f"\n  [{r['llm']}]")
        for d in r['details']:
            print(f"    Query : {d['query'][:45]:45s} → score={d['generation']['score_global']}/10 | lat={d['t_generation']:.1f}s")

    return rapports

rapports_llm = comparer_llms()



 COMPARAISON DES LLMs (1 modèles benchmarkés)
LLM                            | Score/10 | Syntaxe  | Lat(s)  | P@K  
-----------------------------------------------------------------
Mistral-7B-Instruct-v0.1       |     6.69 |     75%  |    57.5 | 1.000

─────────────────────────────────────────────────────────────────
 DÉTAIL PAR QUERY
─────────────────────────────────────────────────────────────────

  [Mistral-7B-Instruct-v0.1]
    Query : Détecter un ransomware qui chiffre les fichie → score=0.0/10 | lat=90.2s
    Query : Trouver un trojan qui ouvre un reverse shell  → score=9.4/10 | lat=43.9s
    Query : Identifier un spyware qui enregistre les frap → score=9.12/10 | lat=32.9s
    Query : Détecter un dropper utilisant PowerShell      → score=8.25/10 | lat=63.0s


## Cellule 17 — Clustering des règles générées
> Génère plusieurs règles puis les regroupe par similarité sémantique.

In [ ]:
# ================================================================
# CELLULE 17 — Clustering des règles générées
# ================================================================

def generer_et_clusterer(index, documents, bm25, n_clusters=6):
    """
    Génère une règle pour chaque query de test
    puis regroupe les règles par similarité.
    """
    from sklearn.cluster import KMeans
    from sklearn.feature_extraction.text import TfidfVectorizer

    queries_cluster = [
        "Détecter un ransomware qui chiffre les fichiers avec AES",
        "Ransomware supprimant les shadow copies",
        "Ransomware avec extension .locked",
        "Trojan ouvrant un reverse shell",
        "Trojan modifiant le registre Windows",
        "Worm se propageant via SMB",
        "Worm copiant sur clé USB",
        "Spyware enregistrant les touches clavier",
        "Spyware capturant l'écran",
        "Dropper utilisant PowerShell",
        "Dropper cachant payload dans ZIP",
        "Rootkit masquant processus via DKOM",
    ]

    print(f"[Clustering] Génération de {len(queries_cluster)} règles...")
    regles_generees = []

    for i, query in enumerate(queries_cluster, 1):
        print(f"  [{i}/{len(queries_cluster)}] {query[:50]}...")
        try:
            res = pipeline_rag(query, index, documents, bm25, rag_type="hybride")
            if res['regle']:
                regles_generees.append({
                    "query"      : query,
                    "regle"      : res['regle'],
                    "description": res['description'],
                    "rag_type"   : res['rag_type'],
                })
        except Exception as e:
            print(f"    Erreur : {e}")

    if len(regles_generees) < 3:
        print("Pas assez de règles générées pour le clustering.")
        return {}

    # Vectorisation TF-IDF sur description + règle
    textes = [r['description'] + ' ' + r['regle'] for r in regles_generees]
    n_clust = min(n_clusters, len(regles_generees))
    vec     = TfidfVectorizer(max_features=150, ngram_range=(1,2))
    X       = vec.fit_transform(textes).toarray()
    km      = KMeans(n_clusters=n_clust, random_state=42, n_init=10)
    labels  = km.fit_predict(X)

    # Affichage des clusters
    clusters = {}
    for i, (label, r) in enumerate(zip(labels, regles_generees)):
        key = f"Cluster_{label}"
        if key not in clusters:
            clusters[key] = []
        clusters[key].append(r['query'])

    print(f"\n{'='*55}")
    print(f" CLUSTERING — {n_clust} groupes pour {len(regles_generees)} règles")
    print(f"{'='*55}")
    for nom, membres in sorted(clusters.items()):
        print(f"\n  {nom} ({len(membres)} règles) :")
        for q in membres:
            print(f"    • {q}")

    # Sauvegarder les règles générées
    path_out = f"{CACHE_DIR}/regles_generees_cluster.json"
    with open(path_out, 'w', encoding='utf-8') as f:
        json.dump({"regles": regles_generees, "clusters": clusters, "labels": labels.tolist()}, f,
                  indent=2, ensure_ascii=False)
    print(f"\n  Sauvegardé : {path_out}")

    return clusters, regles_generees

index, documents, bm25 = initialiser()
clusters, regles = generer_et_clusterer(index, documents, bm25, n_clusters=6)


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Clustering] Génération de 12 règles...
  [1/12] Détecter un ransomware qui chiffre les fichiers av...

[RAG] Query    : Détecter un ransomware qui chiffre les fichiers avec AES
[RAG] Type     : hybride
[RAG] Récupéré : 3 documents (latence: 0.01s)
       → SYN_RAN_004 | ransomware | score=0.766
       → RAN_001 | ransomware | score=0.740
       → SYN_RAN_008 | ransomware | score=0.561
[LLM] Génération en cours...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[LLM] Génération terminée (latence: 43.54s)
  [2/12] Ransomware supprimant les shadow copies...

[RAG] Query    : Ransomware supprimant les shadow copies
[RAG] Type     : hybride
[RAG] Récupéré : 3 documents (latence: 0.01s)
       → RAN_004 | ransomware | score=1.000
       → SYN_RAN_007 | ransomware | score=0.239
       → SYN_RAN_004 | ransomware | score=0.231
[LLM] Génération en cours...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[LLM] Génération terminée (latence: 48.63s)
  [10/12] Dropper utilisant PowerShell...

[RAG] Query    : Dropper utilisant PowerShell
[RAG] Type     : hybride
[RAG] Récupéré : 3 documents (latence: 0.01s)
       → DRO_001 | dropper | score=1.000
       → SYN_DRO_037 | dropper | score=0.398
       → SYN_DRO_039 | dropper | score=0.377
[LLM] Génération en cours...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[LLM] Génération terminée (latence: 38.86s)
  [11/12] Dropper cachant payload dans ZIP...

[RAG] Query    : Dropper cachant payload dans ZIP
[RAG] Type     : hybride
[RAG] Récupéré : 3 documents (latence: 0.01s)
       → DRO_004 | dropper | score=1.000
       → SYN_DRO_034 | dropper | score=0.685
       → DRO_003 | dropper | score=0.422
[LLM] Génération en cours...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[LLM] Génération terminée (latence: 32.27s)
  [12/12] Rootkit masquant processus via DKOM...

[RAG] Query    : Rootkit masquant processus via DKOM
[RAG] Type     : hybride
[RAG] Récupéré : 3 documents (latence: 0.01s)
       → ROO_003 | rootkit | score=1.000
       → SYN_ROO_045 | rootkit | score=0.635
       → ROO_002 | rootkit | score=0.269
[LLM] Génération en cours...
[LLM] Génération terminée (latence: 36.50s)

 CLUSTERING — 6 groupes pour 12 règles

  Cluster_0 (2 règles) :
    • Trojan ouvrant un reverse shell
    • Spyware capturant l'écran

  Cluster_1 (2 règles) :
    • Worm se propageant via SMB
    • Worm copiant sur clé USB

  Cluster_2 (4 règles) :
    • Trojan modifiant le registre Windows
    • Spyware enregistrant les touches clavier
    • Dropper cachant payload dans ZIP
    • Rootkit masquant processus via DKOM

  Cluster_3 (2 règles) :
    • Détecter un ransomware qui chiffre les fichiers avec AES
    • Ransomware supprimant les shadow copies

  Cluster_4 (1 règles) 

## Cellule 18 — Rapport global de synthèse
> Génère un rapport complet de tous les benchmarks pour le livrable.

In [ ]:
# ================================================================
# CELLULE 18 — Rapport global de synthèse
# ================================================================

import glob, json
from datetime import datetime

def generer_rapport_global():
    """Compile tous les résultats en un rapport lisible"""
    rapport = {
        "date"          : datetime.now().isoformat(),
        "projet"        : "YARA RAG Pipeline — FST Tanger 2026",
        "groupe"        : "GRPP-2",
        "llms"          : [],
        "rag_types"     : {},
        "dataset"       : {},
        "conclusion"    : {}
    }

    # Dataset
    try:
        with open(DATASET_PATH) as f:
            data = json.load(f)
        from collections import Counter
        rapport["dataset"] = {
            "total"     : len(data),
            "familles"  : dict(Counter(r['famille'] for r in data)),
            "sources"   : dict(Counter(r['source']  for r in data)),
        }
    except: pass

    # LLMs
    for f in sorted(glob.glob(f"{CACHE_DIR}/benchmark_llm_*.json")):
        with open(f) as fp:
            r = json.load(fp)
        rapport["llms"].append({
            "nom"           : r['llm'],
            "score_moyen"   : r['score_moyen'],
            "syntaxe"       : r['syntaxe_valide'],
            "latence_moy"   : r['latence_gen_moy'],
            "precision_k"   : r['precision_k_moy'],
        })

    # RAG types
    rag_path = f"{CACHE_DIR}/benchmark_rag_types.json"
    if os.path.exists(rag_path):
        with open(rag_path) as f:
            rapport["rag_types"] = json.load(f)

    # Conclusion automatique
    if rapport["llms"]:
        best_llm = max(rapport["llms"], key=lambda x: x['score_moyen'])
        rapport["conclusion"]["meilleur_llm"]   = best_llm['nom']
        rapport["conclusion"]["score_llm"]      = best_llm['score_moyen']
    if rapport["rag_types"]:
        best_rag = max(rapport["rag_types"], key=lambda x: rapport["rag_types"][x].get('score_global',0))
        rapport["conclusion"]["meilleur_rag"]   = best_rag
        rapport["conclusion"]["score_rag"]      = rapport["rag_types"][best_rag].get('score_global', 0)

    # Sauvegarder
    path_out = f"{CACHE_DIR}/rapport_final.json"
    with open(path_out, 'w', encoding='utf-8') as f:
        json.dump(rapport, f, indent=2, ensure_ascii=False)

    # Affichage lisible
    print("="*65)
    print(f" RAPPORT GLOBAL — {rapport['projet']}")
    print(f" Date : {rapport['date'][:10]}")
    print("="*65)

    print(f"\n DATASET")
    d = rapport['dataset']
    print(f"  Total règles : {d.get('total', 'N/A')}")
    if 'sources' in d:
        for src, cnt in d['sources'].items():
            print(f"  {src:15s} : {cnt}")

    print(f"\n BENCHMARK LLMs")
    if rapport["llms"]:
        print(f"  {'LLM':30s} | {'Score':6s} | {'Syntaxe':8s} | {'Latence':8s}")
        print("  " + "-"*60)
        for llm in rapport["llms"]:
            print(f"  {llm['nom']:30s} | {llm['score_moyen']:6.2f} | {llm['syntaxe']*100:6.0f}%  | {llm['latence_moy']:6.1f}s")
    else:
        print("  Aucun LLM benchmarké (exécutez Cellule 15 d'abord)")

    print(f"\n BENCHMARK RAG TYPES")
    if rapport["rag_types"]:
        print(f"  {'Type':10s} | {'Score':6s} | {'P@K':5s} | {'MRR':5s} | {'Lat_ret':8s}")
        print("  " + "-"*50)
        for rt, m in rapport["rag_types"].items():
            print(f"  {rt:10s} | {m.get('score_global',0):6.2f} | {m.get('precision_k',0):5.3f} | {m.get('mrr',0):5.3f} | {m.get('latence_ret_s',0):6.2f}s")
    else:
        print("  Aucun benchmark RAG (exécutez Cellule 17 d'abord)")

    print(f"\n CONCLUSION")
    c = rapport["conclusion"]
    if c:
        print(f"  Meilleur LLM : {c.get('meilleur_llm','N/A')} — score {c.get('score_llm','N/A')}/10")
        print(f"  Meilleur RAG : {c.get('meilleur_rag','N/A')} — score {c.get('score_rag','N/A')}/10")

    print(f"\n  Rapport sauvegardé : {path_out}")
    print("="*65)

    return rapport

rapport_global = generer_rapport_global()


 RAPPORT GLOBAL — YARA RAG Pipeline — FST Tanger 2026
 Date : 2026-06-11

 DATASET
  Total règles : 80
  manuel          : 30
  synthetique     : 50

 BENCHMARK LLMs
  LLM                            | Score  | Syntaxe  | Latence 
  ------------------------------------------------------------
  Mistral-7B-Instruct-v0.1       |   6.69 |     75%  |   57.5s

 BENCHMARK RAG TYPES
  Type       | Score  | P@K   | MRR   | Lat_ret 
  --------------------------------------------------
  classique  |   6.48 | 1.000 | 1.000 |   0.01s
  hybride    |   6.68 | 1.000 | 1.000 |   0.01s
  rerank     |   9.12 | 0.917 | 1.000 |   3.17s
  agentic    |   7.90 | 1.000 | 1.000 |   0.02s

 CONCLUSION
  Meilleur LLM : Mistral-7B-Instruct-v0.1 — score 6.69/10
  Meilleur RAG : rerank — score 9.12/10

  Rapport sauvegardé : /content/yara_rag/cache/rapport_final.json


##  Cellule 19 — Interface Gradio



In [ ]:
# ================================================================
# CELLULE 19 — Interface Gradio
# ================================================================

# Initialisation au démarrage de l'interface
print("[Init] Initialisation du pipeline...")
index, documents, bm25 = initialiser()
print(f"[Init] Prêt — {len(documents)} documents indexés")


# ---- Fonctions Gradio ----------------------------------------

def gradio_generer(query, rag_type):
    """Génère une règle YARA et retourne les composants pour l'interface"""
    if not query.strip():
        return "", "", "", ""
    try:
        res = pipeline_rag(query, index, documents, bm25, rag_type=rag_type)

        print(f"[DEBUG] regle={len(res['regle'])} chars")   # ← debug
        print(f"[DEBUG] raw={res['raw'][:200] if res['raw'] else 'None'}")

        # Formater les documents récupérés
        docs_info = ""
        for i, d in enumerate(res['docs_recuperes'], 1):
            docs_info += (
                f"**[{i}] {d['id']}** — Famille: `{d['famille']}` | "
                f"Danger: `{d['danger']}` | Score: `{d['score']:.3f}`\n"
                f"*{d['description'][:120]}...*\n\n"
            )

        latences = (
            f"Retrieval : `{res['t_retrieval']:.2f}s` | "
            f"Génération : `{res['t_generation']:.2f}s` | "
            f"Type RAG : `{res['rag_type']}`"
        )

        regle_affichee = res['regle'] if res['regle'] else res['raw'] or "Aucune règle générée."

        return regle_affichee, res['description'], res['reference'], docs_info, latences

    except Exception as e:
        return f"ERREUR : {e}", "", "", "", ""


def gradio_benchmark():
    """Lance le benchmark des 4 types de RAG"""
    try:
        rapport = benchmark_rag_types(index, documents, bm25)
        lignes  = ["| Type | P@k | MRR | Syntaxe | Complétude | Latence (s) |",
                   "|------|-----|-----|---------|------------|------------|"]
        for rt, m in rapport.items():
            lignes.append(
                f"| {rt} | {m['precision_k']} | {m['mrr']} | "
                f"{m['syntaxe_valide']} | {m['completude']} | {m['latence_ret_s']}s |"
            )
        return "\n".join(lignes)
    except Exception as e:
        return f"Erreur benchmark : {e}"


def gradio_ingerer_pdf(fichier):
    """Ingère un PDF dans la base de connaissances"""
    global index, documents, bm25
    if fichier is None:
        return "Aucun fichier sélectionné."
    try:
        index, documents, bm25 = ingerer_pdf(fichier.name, index, documents, bm25)
        return f"PDF ingéré. Total documents : {len(documents)}"
    except Exception as e:
        return f"Erreur : {e}"


# ---- Construction de l'interface Gradio ----------------------

EXAMPLES = [
    ["Détecter un ransomware qui chiffre les fichiers avec AES-256",      "hybride"],
    ["Trouver un trojan qui ouvre un reverse shell via PowerShell",        "rerank"],
    ["Identifier un worm se propageant via SMB avec EternalBlue",          "agentic"],
    ["Détecter un spyware qui enregistre les frappes clavier",            "classique"],
    ["Trouver un dropper qui télécharge un payload via WebClient",         "hybride"],
    ["Identifier un rootkit qui cache ses processus via SSDT hook",        "agentic"],
]

with gr.Blocks(
    title="YARA RAG Pipeline",
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="slate"),
) as demo:

    gr.Markdown("""
    # YARA RAG Pipeline
    ### Génération automatique de règles YARA par Retrieval-Augmented Generation
    **FST Tanger 2026** — GRPP-2
    """)

    # ---- Onglet 1 : Génération --------------------------------
    with gr.Tab("🔧 Génération de règle"):
        with gr.Row():
            with gr.Column(scale=2):
                query_input = gr.Textbox(
                    label="📝 Décrivez la menace à détecter",
                    placeholder="Ex: Détecter un ransomware qui chiffre les fichiers avec AES-256...",
                    lines=3
                )
                rag_type_input = gr.Radio(
                    choices=["classique", "hybride", "rerank", "agentic"],
                    value="hybride",
                    label="⚙️ Type de RAG",
                    info="hybride = recommandé | agentic = adaptatif | rerank = meilleure précision"
                )
                gen_btn = gr.Button(" Générer la règle YARA", variant="primary", size="lg")

            with gr.Column(scale=1):
                gr.Markdown("### 📚 Exemples de requêtes")
                gr.Examples(
                    examples=EXAMPLES,
                    inputs=[query_input, rag_type_input],
                    label=""
                )

        latences_output = gr.Markdown(label="")

        with gr.Row():
            with gr.Column():
                regle_output = gr.Code(
                    label="Règle YARA générée",
                    language="python",
                    lines=18
                )
            with gr.Column():
                docs_output = gr.Markdown(label="Documents récupérés")

        with gr.Row():
            desc_output = gr.Textbox(label="📋 Description", lines=3)
            ref_output  = gr.Textbox(label="🔗 Référence",   lines=2)

        gen_btn.click(
              fn=gradio_generer,
              inputs=[query_input, rag_type_input],
              outputs=[regle_output, desc_output, ref_output, docs_output, latences_output],
              show_progress="full"
          )
    # ---- Onglet 2 : Benchmark ---------------------------------
    with gr.Tab("📊 Benchmark RAG"):
        gr.Markdown("""
        Lance le benchmark sur les 6 requêtes de test et compare les 4 stratégies de retrieval.
        ⚠️ **Attention** : cela effectue 24 appels LLM — peut prendre plusieurs minutes.
        """)
        bench_btn    = gr.Button("▶️ Lancer le benchmark", variant="secondary")
        bench_output = gr.Markdown(label="Résultats")
        bench_btn.click(fn=gradio_benchmark, outputs=bench_output)

    # ---- Onglet 3 : Ingestion PDF -----------------------------
    with gr.Tab("📄 Ingestion PDF"):
        gr.Markdown("""
        Ajoutez un PDF (rapport de threat intelligence, documentation YARA, etc.)
        pour enrichir la base de connaissances du pipeline.
        """)
        pdf_input  = gr.File(label="📎 Sélectionner un fichier PDF", file_types=[".pdf"])
        pdf_btn    = gr.Button("📥 Ingérer le PDF", variant="secondary")
        pdf_output = gr.Textbox(label="Statut")
        pdf_btn.click(fn=gradio_ingerer_pdf, inputs=pdf_input, outputs=pdf_output)

    # ---- Onglet 4 : Info système ------------------------------
    with gr.Tab("ℹ️ Info système"):
        info_md = f"""
        ### Configuration active
        | Paramètre | Valeur |
        |-----------|--------|
        | LLM | `{LLM_NAME}` |
        | Embedding | `{EMBED_MODEL_NAME}` |
        | Reranker | `{RERANK_MODEL_NAME}` |
        | Documents indexés | `{len(documents)}` |
        | Top-K retrieval | `{TOP_K}` |
        | GPU disponible | `{torch.cuda.is_available()}` |
        | Cache dir | `{CACHE_DIR}` |

        ### Types de RAG
        - **Classique** : Recherche sémantique dense uniquement
        - **Hybride** : FAISS + BM25 avec fusion des scores (recommandé)
        - **Rerank** : FAISS + CrossEncoder pour re-classer les candidats
        - **Agentique** : Stratégie adaptative (filtre famille / BM25 / hybride selon la requête)
        """
        gr.Markdown(info_md)


# Lancement — share=True génère un lien public accessible depuis l'extérieur de Colab
demo.launch(share=True, debug=False)

[Init] Initialisation du pipeline...
[Init] Prêt — 81 documents indexés
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cc6ecb6b6079bb8e4b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
